In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv

if os.environ['AICREDITS_API_KEY']:
    print("AICredits API Key is set.")
else:
    raise ValueError("AICredits API Key is not set.")

AICredits API Key is set.


In [2]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from  langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## **CHAIN WITH Parallel Chains**

In [3]:
# TASK -1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human", "Please summarize the movie in brief : {input}")])

In [4]:
# TASK - 2 [LLM]
llm_openai = ChatOpenAI(model="gpt-5-mini",
                        base_url="https://api.aicredits.in/v1",
                        api_key=os.environ["AICREDITS_API_KEY"])

In [5]:
# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

In [6]:
# TASK - 4 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text: str) -> dict:
    return {"text": text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

## **Parallel Chain 1**

In [7]:
# TASK -1 [Prompt]
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")])

# TASK - 2 [LLM]
llm_openai = ChatOpenAI(model="gpt-5-mini",
                        base_url="https://api.aicredits.in/v1",
                        api_key=os.environ["AICREDITS_API_KEY"])

# TASK - 3 [Str Parser]
str_parser = StrOutputParser()

chain_linked_in = linkedin_prompt | llm_openai | str_parser

## **Parallel Chain 2**

In [8]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

In [9]:
def insta_chain(text: dict):
    text = text["text"]
    
    # TASK - 1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for Instagram: {text}")])

    # TASK - 2 [LLM]
    llm_openai = ChatOpenAI(model="gpt-5-mini",
                            base_url="https://api.aicredits.in/v1",
                            api_key=os.environ["AICREDITS_API_KEY"])
    
    # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke(text)
    return result
    
insta_chain_runnable = RunnableLambda(insta_chain)


## **Final Orchestration**

In [10]:
final_chain = (
    prompt_template|
    llm_openai|
    str_parser|
    dictionary_maker_runnable|
    RunnableParallel(branches={"linkedin":chain_linked_in, "instagram":insta_chain_runnable})
)

In [11]:
final_chain.invoke("RRR")

{'branches': {'linkedin': '🎬 Watched RRR — a fictionalized, larger-than-life 1920s epic that brings together two legendary revolutionaries: Alluri Sitarama Raju (a dutiful police officer under British rule) and Komaram Bheem (a tribal leader on a mission to rescue a kidnapped girl). They build a deep friendship without knowing each other’s true motives; when those secret missions collide, friendship turns to conflict — and then to a united, explosive fight against colonial oppression.\n\nBeyond the spectacle, music and grand action set pieces, RRR celebrates timeless leadership themes: courage under pressure, loyalty across differences, the cost of conviction, and the power of storytelling to inspire collective resistance.\n\nTakeaways for leaders and teams:\n- Shared purpose can unite unlikely partners.\n- Honest conflict can surface values that strengthen collaboration.\n- Sacrifice and emotional commitment drive lasting change.\n- Great storytelling magnifies impact and mobilizes pe

## **Chain as a Runnable**

In [15]:
# TASK - 1 [Beautify Function]

def beautify(final_response: dict) -> dict:
    linkedin_response = final_response['branches']['linkedin']
    instagram_response = final_response['branches']['instagram']

    return {"linkedin": linkedin_response, "instagram":instagram_response}

beautify_runnable = RunnableLambda(beautify)

# TASK - 2 [Final Chain]
beautified_chain = final_chain | beautify_runnable
beautified_chain.invoke("War - 2019")

{'linkedin': 'Just watched War (2019) — a slick, high-octane action thriller directed by Siddharth Anand that doubles as a study in mentorship and conflict. Hrithik Roshan plays Kabir, a top RAW agent who appears to go rogue, and Tiger Shroff is Khalid, his former protégé sent to bring him in. The globe‑trotting cat‑and‑mouse chase delivers jaw‑dropping stunts, tight fight choreography, striking visuals and a soundtrack that stuck with me long after the credits rolled.\n\nBeyond the spectacle, the film highlights a few leadership themes that resonate in any high‑performance environment:\n- Mentor vs. protégé dynamics — the thin line between guidance and independence.\n- Managing trust and accountability when outcomes deviate from expectations.\n- The payoff of preparation and rehearsal: elaborate action set pieces mirror the discipline and coordination top teams require.\n- The value of storytelling and craft (visuals, choreography, music) in creating memorable experiences.\n\nIf you e